# GAN usage in DEGANN

#### This notebook demonstrates how to create and train a GAN using DEGANN.

**Import necessary libraries:**

In [13]:
import tensorflow as tf
import numpy as np
from scipy.integrate import solve_ivp

from degann.networks.topology.parameter_space import GANParameterSpace
from degann.search_algorithms.grid_search import grid_search
from degann.search_algorithms.random_search import random_search
from degann.search_algorithms.simulated_annealing import simulated_annealing

**Prepare data for training:**

In [14]:
def lf_ode1(t, y):
    return -3 * y

sol = solve_ivp(lf_ode1, [0, 5], [1], t_eval=np.linspace(0, 5, 400))
X, y = sol.t.reshape(-1, 1), sol.y[0].reshape(-1, 1)

**Define and create paramter object for GAN:**

In [16]:
GAN_params = GANParameterSpace(
    gen_input_size=1,
    gen_output_size=1,
    gen_layer_sizes=[8, 16, 24, 32],
    gen_min_depth=2,
    gen_max_depth=4,
    gen_activation_funcs=["relu", "tanh", "swish"],
    gen_out_activation="linear",

    disc_layer_sizes=[8, 16, 24, 32],
    disc_min_depth=2,
    disc_max_depth=4,
    disc_activation_funcs=["relu", "tanh"],

    gen_optimizers=["Adam", "RMSprop"],
    disc_optimizers=["Adam", "RMSprop"],
    gen_loss_funcs=["MeanSquaredError"],
    disc_loss_funcs=["MeanSquaredError"],
    epochs=[50, 100, 150],
)

In [6]:
GAN_params_light = GANParameterSpace(
    gen_input_size=1,
    gen_output_size=1,
    gen_layer_sizes=[16],
    gen_min_depth=3,
    gen_max_depth=6,
    gen_activation_funcs=["relu"],
    gen_out_activation="linear",

    disc_layer_sizes=[32],
    disc_min_depth=2,
    disc_max_depth=4,
    disc_activation_funcs=["tanh"],

    gen_optimizers=["Adam"],
    disc_optimizers=["Adam"],
    gen_loss_funcs=["MeanSquaredError"],
    disc_loss_funcs=["MeanSquaredError"],
    epochs=[100],
)

**Grid search**

In [7]:
grid_loss, grid_epoch, grid_loss_name, grid_optimizer, grid_nn = grid_search(
    data=(X, y),
    params=GAN_params_light,
    verbose=True,
)

1/12 2025-12-24 15:07:33
2/12 2025-12-24 15:08:58
3/12 2025-12-24 15:10:31
4/12 2025-12-24 15:12:15
5/12 2025-12-24 15:13:43
6/12 2025-12-24 15:15:19
7/12 2025-12-24 15:17:06
8/12 2025-12-24 15:18:46
9/12 2025-12-24 15:20:35
10/12 2025-12-24 15:22:31
11/12 2025-12-24 15:24:15
12/12 2025-12-24 15:26:09


In [8]:
print(grid_loss, grid_nn)

0.24873074889183044 {'net_type': 'TFGAN', 'name': 'net', 'config': {'gen_input_size': 1, 'gen_output_size': 1, 'gen_block_sizes': [16, 16, 16], 'gen_activation_funcs': ['relu', 'relu', 'relu', 'linear'], 'gen_out_activation': 'linear', 'disc_block_sizes': [32, 32], 'disc_activation_funcs': ['tanh', 'tanh', 'linear'], 'disc_out_activation': 'linear', 'gen_optimizer': 'Adam', 'disc_optimizer': 'Adam', 'gen_loss_func': 'MeanSquaredError', 'disc_loss_func': 'MeanSquaredError', 'num_epoch': 100}, 'generator': {'net_type': 'TFGenerator', 'name': 'TFGenerator', 'input_size': 1, 'block_sizes': [16, 16, 16], 'output_size': 1, 'activation_funcs': ['relu', 'relu', 'relu', 'linear'], 'out_activation': 'linear', 'layer': [{'shape': 16, 'inp_size': 1, 'weights': [[0.10003358125686646, -1.1780831813812256, 1.0800610780715942, 0.18612432479858398, -2.486426591873169, -0.31049343943595886, -0.884973406791687, -1.097836971282959, 2.1240921020507812, 1.748468041419983, -0.5079635381698608, 2.122292280197

**Random search**

In [9]:
random_loss, random_epoch, random_loss_name, random_optimizer, random_nn = random_search(
    data=(X, y),
    params=GAN_params,
    iterations=10,
    verbose=True
)

1/10 2025-12-24 15:28:13
2/10 2025-12-24 15:30:05
3/10 2025-12-24 15:31:26
4/10 2025-12-24 15:33:33
5/10 2025-12-24 15:35:16
6/10 2025-12-24 15:36:00
7/10 2025-12-24 15:36:43
8/10 2025-12-24 15:38:27
9/10 2025-12-24 15:39:14
10/10 2025-12-24 15:39:51


In [10]:
print(random_loss, random_nn)

0.22376255691051483 {'net_type': 'TFGAN', 'name': 'net', 'config': {'gen_input_size': 1, 'gen_output_size': 1, 'gen_block_sizes': [32, 16, 16], 'gen_activation_funcs': ['tanh', 'swish', 'relu', 'linear'], 'gen_out_activation': 'linear', 'disc_block_sizes': [32, 16, 24], 'disc_activation_funcs': ['relu', 'tanh', 'relu', 'linear'], 'disc_out_activation': 'linear', 'gen_optimizer': 'Adam', 'disc_optimizer': 'RMSprop', 'gen_loss_func': 'MeanSquaredError', 'disc_loss_func': 'MeanSquaredError', 'num_epoch': 100}, 'generator': {'net_type': 'TFGenerator', 'name': 'TFGenerator', 'input_size': 1, 'block_sizes': [32, 16, 16], 'output_size': 1, 'activation_funcs': ['tanh', 'swish', 'relu', 'linear'], 'out_activation': 'linear', 'layer': [{'shape': 32, 'inp_size': 1, 'weights': [[0.1067328155040741, -1.1774396896362305, 1.0904957056045532, 0.1225450411438942, -2.3995206356048584, -0.32828059792518616, -0.8648504614830017, -1.1041443347930908, 2.3337619304656982, 1.6553298234939575, -0.5196750164031

**Simulated annealing algorithm**

In [11]:
sim_loss, sim_config, sim_nn, sim_iteration = simulated_annealing(
    data=(X, y),
    params=GAN_params,
    max_iter=10,
    threshold=0.25,
    verbose=True
)

1/10 2025-12-24 15:43:10
2/10 2025-12-24 15:45:10
3/10 2025-12-24 15:47:28


In [12]:
print(sim_loss, sim_nn)

0.2498936578631401 {'net_type': 'TFGAN', 'name': 'net', 'config': {'gen_input_size': 1, 'gen_output_size': 1, 'gen_block_sizes': [32, 24, 16], 'gen_activation_funcs': ['tanh', 'swish', 'tanh', 'linear'], 'gen_out_activation': 'linear', 'disc_block_sizes': [8, 8, 16, 8], 'disc_activation_funcs': ['tanh', 'relu', 'tanh', 'tanh', 'linear'], 'disc_out_activation': 'linear', 'gen_optimizer': 'Adam', 'disc_optimizer': 'Adam', 'gen_loss_func': 'MeanSquaredError', 'disc_loss_func': 'MeanSquaredError', 'num_epoch': 150}, 'generator': {'net_type': 'TFGenerator', 'name': 'TFGenerator', 'input_size': 1, 'block_sizes': [32, 24, 16], 'output_size': 1, 'activation_funcs': ['tanh', 'swish', 'tanh', 'linear'], 'out_activation': 'linear', 'layer': [{'shape': 32, 'inp_size': 1, 'weights': [[0.14582188427448273, -1.1735401153564453, 1.1403601169586182, 0.11474201828241348, -2.616957902908325, -0.3323388993740082, -0.8938468098640442, -1.0670404434204102, 2.1751198768615723, 1.663751244544983, -0.553965151